# Prophet - Trend i Sezonalnost

Pristup: treniramo Prophet na agregatnoj dnevnoj prodaji po prodavnici (54 modela).
Vadimo `trend`, `weekly` i `yearly` komponente kao features za LightGBM.

Output: `data/processed/prophet_features.parquet`

In [ ]:
import sys, subprocess
print('Python:', sys.executable)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'prophet', '-q'], check=True)
print('Instalacija gotova.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import warnings
import logging
import os

warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

DATA      = '../data/raw/'
PROCESSED = '../data/processed/'

## 1. Priprema podataka

In [ ]:
train = pd.read_csv(DATA + 'train.csv', parse_dates=['date'])

# Agregatna dnevna prodaja po prodavnici (sumiramo sve kategorije)
store_daily = (
    train
    .groupby(['store_nbr', 'date'])['sales']
    .sum()
    .reset_index()
    .rename(columns={'date': 'ds', 'sales': 'y'})
)

stores = store_daily['store_nbr'].unique()
print(f'Broj prodavnica: {len(stores)}')
print(f'Period: {store_daily.ds.min().date()} do {store_daily.ds.max().date()}')
store_daily.head()

## 2. Treniranje Prophet modela (54 modela)

Za svaku prodavnicu: fit Prophet, ekstraktujemo trend + weekly + yearly komponente.

In [ ]:
def fit_prophet(store_df):
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.05,
    )
    m.fit(store_df[['ds', 'y']])
    forecast = m.predict(store_df[['ds']])
    return forecast[['ds', 'trend', 'weekly', 'yearly', 'yhat']]


all_forecasts = []
total = len(stores)

for i, store in enumerate(sorted(stores), 1):
    df_store = store_daily[store_daily['store_nbr'] == store].copy()
    forecast = fit_prophet(df_store)
    forecast['store_nbr'] = store
    all_forecasts.append(forecast)
    if i % 10 == 0 or i == total:
        print(f'  {i}/{total} prodavnica gotovo...')

prophet_df = pd.concat(all_forecasts, ignore_index=True)
prophet_df = prophet_df.rename(columns={
    'ds':     'date',
    'trend':  'prophet_trend',
    'weekly': 'prophet_weekly',
    'yearly': 'prophet_yearly',
    'yhat':   'prophet_yhat',
})

print(f'\nProphet features shape: {prophet_df.shape}')
prophet_df.head()

## 3. Vizualizacija - primjer za jednu prodavnicu

In [ ]:
store_ex = 1
ex_data     = store_daily[store_daily['store_nbr'] == store_ex]
ex_forecast = prophet_df[prophet_df['store_nbr'] == store_ex]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Stvarna vs Prophet predikcija
axes[0].plot(ex_data['ds'], ex_data['y'], linewidth=0.7, color='steelblue', label='Stvarna prodaja')
axes[0].plot(ex_forecast['date'], ex_forecast['prophet_yhat'], linewidth=1, color='darkorange', linestyle='--', label='Prophet yhat')
axes[0].set_title(f'Prodavnica {store_ex} - stvarna vs Prophet predikcija')
axes[0].legend()

# Trend
axes[1].plot(ex_forecast['date'], ex_forecast['prophet_trend'], color='green', linewidth=1)
axes[1].set_title('Trend komponenta')

# Weekly sezonalnost
axes[2].plot(ex_forecast['date'], ex_forecast['prophet_weekly'], color='purple', linewidth=0.7)
axes[2].set_title('Weekly sezonalnost')

plt.tight_layout()
plt.show()

## 4. Merge sa postojecim feature datasetom

In [ ]:
train_df = pd.read_parquet(PROCESSED + 'train_features.parquet')
val_df   = pd.read_parquet(PROCESSED + 'val_features.parquet')

prophet_features = ['date', 'store_nbr', 'prophet_trend', 'prophet_weekly', 'prophet_yearly', 'prophet_yhat']

train_df = train_df.merge(prophet_df[prophet_features], on=['date', 'store_nbr'], how='left')
val_df   = val_df.merge(prophet_df[prophet_features],   on=['date', 'store_nbr'], how='left')

print(f'Train shape: {train_df.shape}')
print(f'Val shape:   {val_df.shape}')
print(f'Missing prophet features in train: {train_df["prophet_trend"].isnull().sum()}')
print(f'Missing prophet features in val:   {val_df["prophet_trend"].isnull().sum()}')

## 5. Snimanje

In [ ]:
train_df.to_parquet(PROCESSED + 'train_features.parquet', index=False)
val_df.to_parquet(PROCESSED   + 'val_features.parquet',   index=False)
prophet_df.to_parquet(PROCESSED + 'prophet_features.parquet', index=False)

print('Snimljeno:')
print(f'  train_features.parquet  ({train_df.shape[0]:,} redova, {train_df.shape[1]} kolona)')
print(f'  val_features.parquet    ({val_df.shape[0]:,} redova, {val_df.shape[1]} kolona)')
print(f'  prophet_features.parquet')

## 6. Koliko Prophet features pomazu LightGBM-u?

Brza provjera korelacije novih features sa targetom.

In [ ]:
prophet_cols = ['prophet_trend', 'prophet_weekly', 'prophet_yearly', 'prophet_yhat']
corr = train_df[prophet_cols + ['sales_log']].corr()['sales_log'].drop('sales_log').sort_values()

print('Korelacija Prophet features sa log1p(sales):')
print(corr.to_string())